<a href="https://colab.research.google.com/github/ArhamUsman/AI-LAB/blob/main/k241016_Assignment_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Task 1**
-
Consider a standard deck of 52 cards.
1. What is the probability of drawing a red card
(hearts or diamonds)?

2. Now, given that you have drawn a red card, what is the probability that the card is a heart?

3. Given that the drawn card is a face card (Jack, Queen, King), what is the probability that it is a diamond?
3. Given that the card drawn is a face card, calculate the probability that it is also a spade or a queen.

In [ ]:
import random

def prob(total, expected):
  trials=10000
  c=0
  for _ in range(trials):
    card=random.choice(total)
    if card in expected:
      c+=1
  return round(c/trials,2)

suits=['hearts', 'diamonds', 'clubs', 'spades']
ranks=['2', '3', '4', '5', '6', '7', '8', '9', 'jack', 'queen', 'king', 'ace']
deck=[(rank, suit) for rank in ranks for suit in suits]

red_cards=[(rank, suit) for rank in ranks for suit in ['hearts', 'diamonds']]

print("1. Prob of reb cards:", prob(deck, red_cards))

hearts=[(rank, suit) for rank in ranks for suit in ['hearts']]

print("2. Prob of hearts given red card: ", prob(red_cards, hearts))

face_cards=[(rank, suit) for rank in ['jack', 'queen', 'king'] for suit in suits]
diamonds=[(rank, suit) for rank in ranks for suit in ['diamonds']]

print("3. Prob of diamond given face card: ", prob(face_cards, diamonds))

spade={(rank, suit) for rank in ranks for suit in ['spade']}
queen={(rank, suit) for rank in ['queen'] for suit in suits}

print("4. Prob of spade or queen given face card: ", prob(face_cards, list(spade.union(queen))))


1. Prob of reb cards: 0.5
2. Prob of hearts given red card:  0.5
3. Prob of diamond given face card:  0.25
4. Prob of spade or queen given face card:  0.34


**Task 2**
-
You are given the following Bayesian Network
structure to model student exam performance:

Dependencies:
- Intelligence,
- StudyHours,
- and Difficulty directly affect Grade.
- Grade directly affects Pass.

Nodes:
- Intelligence (I) — {High, Low}
- StudyHours (S) — {Sufficient, Insufficient}
- Difficulty (D) — {Hard, Easy}
- Grade (G) — {A, B, C}
- Pass (P) — {Yes, No}

Prior Probabilities:
- P(Intelligence = High) = 0.7, P(Intelligence = Low) = 0.3
- P(StudyHours = Sufficient) = 0.6, P(StudyHours = Insufficient) = 0.4
- P(Difficulty = Hard) = 0.4, P(Difficulty = Easy) = 0.6

Conditional Probabilities (examples):

P(Grade | Intelligence, StudyHours, Difficulty): (Assume your own valid values, e.g., students with High intelligence, Sufficient
study hours, and Easy difficulty are most likely to get A)
- P(Pass | Grade):
- P(Pass = Yes | Grade = A) = 0.95
- P(Pass = Yes | Grade = B) = 0.80
- P(Pass = Yes | Grade = C) = 0.50

Tasks to do
1. Construct the Bayesian Network structure diagram showing all dependencies.
2. Define the complete Conditional Probability Tables (CPTs) for all nodes.
3. Implement the Bayesian Network using Python (pgmpy or equivalent).
4. Perform inference using Variable Elimination to answer:
5. What is the probability that the student passes the exam, given: StudyHours = Sufficient, Difficulty = Hard
6. What is the probability that the student has High Intelligence, given: Pass = Yes

In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

#dependencies
model= DiscreteBayesianNetwork([
    ('intelligence', 'grade'),
    ('study_hours', 'grade'),
    ('difficulty', 'grade'),
    ('grade', 'pass')
])

#values with probs
cpd_i=TabularCPD('intelligence', 2, [[0.7], [0.3]], state_names={'intelligence':['high', 'low']})
cpd_s=TabularCPD('study_hours', 2, [[0.6], [0.4]], state_names={'study_hours':['sufficient', 'insufficient']})
cpd_d=TabularCPD('difficulty', 2, [[0.4], [0.6]], state_names={'difficulty':['hard', 'easy']})
cpd_g=TabularCPD('grade', 3, [
    [0.9, 0.7, 0.6, 0.3, 0.4, 0.2, 0.1, 0.05],
    [0.08, 0.2, 0.3, 0.4, 0.4, 0.3, 0.2, 0.15],
    [0.02, 0.1, 0.1, 0.3, 0.2, 0.5, 0.7, 0.8]
  ], evidence=['intelligence', 'study_hours', 'difficulty'], evidence_card=[2,2,2],
  state_names={
      'grade': ['A', 'B', 'C'],
      'intelligence': ['high', 'low'],
      'study_hours': ['sufficient', 'insufficient'],
      'difficulty': ['hard', 'easy']
  })

cpd_p = TabularCPD('pass', 2, [
    [0.95, 0.80, 0.50], # Yes
    [0.05, 0.20, 0.50]  # No
], evidence=['grade'], evidence_card=[3],
   state_names={'pass': ['yes', 'no'], 'grade': ['A', 'B', 'C']})

model.add_cpds(cpd_i, cpd_s, cpd_d, cpd_g, cpd_p)
infer = VariableElimination(model)

q1 = infer.query(variables=['pass'], evidence={'study_hours': 'sufficient', 'difficulty': 'hard'})
print(q1)
print()
q2 = infer.query(variables=['intelligence'], evidence={'pass': 'yes'})
print(q2)

+-----------+-------------+
| pass      |   phi(pass) |
+===========+=============+
| pass(yes) |      0.8903 |
+-----------+-------------+
| pass(no)  |      0.1097 |
+-----------+-------------+

+--------------------+---------------------+
| intelligence       |   phi(intelligence) |
+====================+=====================+
| intelligence(high) |              0.7490 |
+--------------------+---------------------+
| intelligence(low)  |              0.2510 |
+--------------------+---------------------+


**Task 3**
-
You are tasked with building a Bayesian Network to predict the likelihood of a
disease (e.g., Flu or Cold) based on the presence of various symptoms, including
fever, cough, fatigue, and chills.

The network consists of the following nodes:
- Symptoms: Fever, Cough, Fatigue, Chills
- Disease: Flu, Cold

Network Structure:

The Disease node influences the symptoms: Fever, Cough, Fatigue, and Chills.

The Symptoms nodes are observed, and they are conditionally dependent on the Disease
node.

Prior Probabilities:

P(Disease):
1. P(Flu) = 0.3
2. P(Cold) = 0.7

P(Symptoms | Disease): (You are to define the following conditional probabilities
based on assumptions )
P(Fever | Disease):
1. P(Fever = Yes | Flu) = 0.9
2. P(Fever = Yes | Cold) = 0.5
3. P(Fever = No | Flu) = 0.1
4. P(Fever = No | Cold) = 0.5

P(Cough | Disease):
1. P(Cough = Yes | Flu) = 0.8
2. P(Cough = Yes | Cold) = 0.6
3. P(Cough = No | Flu) = 0.2
4. P(Cough = No | Cold) = 0.4

P(Fatigue | Disease):
1. P(Fatigue = Yes | Flu) = 0.7
2. P(Fatigue = Yes | Cold) = 0.3
3. P(Fatigue = No | Flu) = 0.3
4. P(Fatigue = No | Cold) = 0.7

P(Chills | Disease):
1. P(Chills = Yes | Flu) = 0.6
2. P(Chills = Yes | Cold) = 0.4
3. P(Chills = No | Flu) = 0.4
4. P(Chills = No | Cold) = 0.6

Construct the Bayesian Network:

Draw the structure of the Bayesian Network based on the dependencies given
above.

Define the Conditional Probability Tables (CPTs)

Define reasonable CPTs for all nodes (Disease, Fever, Cough, Fatigue, and Chills)
based on the given probabilities.

You may assume the probabilities if necessary.

Perform Inference Task 1:

Use the Bayesian Network to calculate the posterior probability of the Disease (Flu
or Cold), given the following symptoms:
- Fever = Yes
- Cough = Yes

Perform Inference Task 2:

Add a new symptom: Chills.

Update the Bayesian Network to compute the posterior probability of the Disease (Flu
or Cold), given the following symptoms:
- Fever = Yes
- Cough = Yes
- Chills = Yes

Perform Inference Task 3:

Given that the disease is Flu, calculate the probability that the person also has
Fatigue.  What is P(Fatigue = Yes | Disease = Flu)?

**Task 4**
-
Define a simple Markov Model with three weather states: Sunny, Cloudy, and Rainy.

Create a transition matrix for the weather states, specifying the probabilities of
transitioning from one state to another (e.g., from Sunny to Cloudy, or from Cloudy to
Rainy).

Simulate the weather for the next 10 days, starting with a Sunny day, using the
Markov Model.

Calculate the probability of having at least 3 rainy days over the 10-day period.